# Tutorial 4: Changing Data (Lecture 9, Ch. 20)

So far every tutorial has either built tables or read from them. This one changes what is already in there, with `INSERT`, `UPDATE`, and `DELETE`.

A word of warning before we start. These statements are the ones that can actually damage a database, and there is no undo button. So most of our practice here happens on throwaway copies whose names end in `_demo`, and we tear them down at the end. The one place we touch the real tables is a `DELETE` that we fully expect to **fail**, which is safe precisely because it fails.

Run this notebook after Tutorial 3.

In [ ]:
import pymysql

DB_HOST = "mysql"          # use "localhost" if running outside the jupyter container
DB_PORT = 3306
DB_USER = "root"
DB_PASSWORD = "OSUGoPokes"
DB_NAME = "iem4723_mysql_tutorial"

conn = pymysql.connect(
    host=DB_HOST,
    port=DB_PORT,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME,
)
cursor = conn.cursor()

In [ ]:
def run(label, query, params=None):
    """Run a query and print whatever comes back, under a label."""
    cursor.execute(query, params or ())
    rows = cursor.fetchall()
    print(f"--- {label} ---")
    if not rows:
        print("(no rows)")
    for row in rows:
        print(row)
    print()

## Inserting a Single Row

We met `INSERT` back in the Lecture 7 tutorial when we loaded the CSVs, but there we were looping over a file. Here is the simpler case, one row written out by hand.

Lecture gave two forms. You can supply a value for every column in order, or you can name the columns you are filling. Naming them is the better habit: it still works if someone later adds a column, and anyone reading it can see what each value means without counting commas.

We will build a small `vendor_demo` table to play in rather than touching the real *vendor* table.

In [ ]:
create_vendor_demo_query = """
DROP TABLE IF EXISTS vendor_demo;
CREATE TABLE vendor_demo (
    Vendor_Num VARCHAR(10) PRIMARY KEY,
    Vendor_Name VARCHAR(50),
    Vendor_City VARCHAR(50),
    Vendor_State VARCHAR(2)
);
"""
for statement in create_vendor_demo_query.strip().split(";"):
    if statement.strip():
        cursor.execute(statement)
conn.commit()

# form 1, a value for every column, in the order the columns were created
insert_all_columns_query = """
INSERT INTO vendor_demo
VALUES (%s, %s, %s, %s);
"""
cursor.execute(insert_all_columns_query, ("V9001", "STILLWATER SUPPLY", "STILLWATER", "OK"))

# form 2, naming the columns we are filling
insert_named_columns_query = """
INSERT INTO vendor_demo (Vendor_Num, Vendor_Name, Vendor_State)
VALUES (%s, %s, %s);
"""
cursor.execute(insert_named_columns_query, ("V9002", "BOOMER DISTRIBUTION", "OK"))

conn.commit()
run("vendor_demo after two inserts", "SELECT * FROM vendor_demo;")

Notice what happened to *Vendor_City* in the second row. We never gave it a value, so MySQL filled in `NULL`. That is the `IS NULL` case from Lecture 8 showing up for real, rather than as a hypothetical.

## Copying Rows With INSERT INTO ... SELECT

The second form of `INSERT` takes its rows from a query instead of from a literal list. Whatever the `SELECT` returns gets inserted, so long as the columns line up.

This is genuinely useful here, because it is how we are going to make a safe copy of *customer* to practise `UPDATE` and `DELETE` on.

In [ ]:
create_customer_demo_query = """
DROP TABLE IF EXISTS customer_demo;
CREATE TABLE customer_demo (
    Customer_Num VARCHAR(10) PRIMARY KEY,
    Customer_FirstName VARCHAR(50),
    Customer_LastName VARCHAR(50),
    Customer_City VARCHAR(50),
    Customer_State VARCHAR(2)
);
"""
for statement in create_customer_demo_query.strip().split(";"):
    if statement.strip():
        cursor.execute(statement)

copy_rows_query = """
INSERT INTO customer_demo (Customer_Num, Customer_FirstName, Customer_LastName, Customer_City, Customer_State)
SELECT Customer_Num, Customer_FirstName, Customer_LastName, Customer_City, Customer_State
FROM customer;
"""
cursor.execute(copy_rows_query)
conn.commit()

cursor.execute("SELECT COUNT(*) FROM customer_demo;")
print(f"customer_demo now holds {cursor.fetchone()[0]} rows, copied straight out of customer.")

## Updating One Row

`UPDATE` has three parts: the table, what to set, and which rows to set it on. That third part is the one to pay attention to.

Because *Customer_Num* is the primary key, this predicate can match at most one row. That is the safest kind of `UPDATE` there is.

In [ ]:
update_one_query = """
UPDATE customer_demo
SET Customer_City = %s
WHERE Customer_Num = %s;
"""
cursor.execute(update_one_query, ("TULSA", "C25"))
conn.commit()
print(f"rows changed: {cursor.rowcount}")

run("C25 after the update", "SELECT * FROM customer_demo WHERE Customer_Num = %s;", ("C25",))

*cursor.rowcount* is worth knowing about. It tells you how many rows the statement actually touched, which is the quickest way to confirm an `UPDATE` did what you meant rather than something broader.

## Updating Many Rows

Change the predicate so it matches more than one row, and `UPDATE` happily changes all of them. Nothing warns you. The only difference between this and the previous cell is the `WHERE` clause.

In [ ]:
update_many_query = """
UPDATE customer_demo
SET Customer_State = %s
WHERE Customer_State = %s;
"""
cursor.execute(update_many_query, ("TX", "PA"))
conn.commit()
print(f"rows changed: {cursor.rowcount}")

run("customers now in TX", "SELECT * FROM customer_demo WHERE Customer_State = 'TX';")

## Leaving the WHERE Clause Off

And here is the one to be careful about. Drop the `WHERE` entirely and the `SET` applies to **every row in the table**.

Occasionally that is exactly what you want, which is why SQL allows it without complaint. Far more often it is a mistake someone notices a few seconds too late. Watch the row count.

In [ ]:
update_all_query = """
UPDATE customer_demo
SET Customer_State = %s;
"""
cursor.execute(update_all_query, ("OK",))
conn.commit()
print(f"rows changed: {cursor.rowcount}   <-- every row in the table")

run("customer_demo after the unfiltered update", "SELECT Customer_Num, Customer_LastName, Customer_State FROM customer_demo;")

## Deleting Rows

`DELETE` follows the same shape, and carries the same warning. A `WHERE` clause picks the rows to remove, and leaving it off removes all of them.

In [ ]:
delete_one_query = """
DELETE FROM customer_demo
WHERE Customer_Num = %s;
"""
cursor.execute(delete_one_query, ("C25",))
conn.commit()
print(f"rows deleted: {cursor.rowcount}")

cursor.execute("SELECT COUNT(*) FROM customer_demo;")
print(f"customer_demo now holds {cursor.fetchone()[0]} rows")

## What Happens When Other Rows Depend on This One

Now the interesting part, and the reason the foreign keys from Tutorial 1 matter.

Customer `C25` has orders sitting in *customerorder*. What should the database do if we try to delete that customer? Three answers are defensible: refuse, delete the orders too, or leave the orders pointing at nobody. SQL lets you choose, and you chose when you wrote `ON DELETE` in the `CREATE TABLE` statement.

In Tutorial 1 we chose `ON DELETE RESTRICT`. So let us try the delete on the **real** *customer* table and see what the database says. This is safe to run precisely because it is going to fail.

In [ ]:
delete_real_customer_query = """
DELETE FROM customer
WHERE Customer_Num = %s;
"""

try:
    cursor.execute(delete_real_customer_query, ("C25",))
    conn.commit()
    print("deleted")
except pymysql.err.IntegrityError as e:
    conn.rollback()
    print("MySQL refused the delete, which is exactly what RESTRICT means:")
    print(" ", e)

Nothing was deleted, and nothing was damaged. `RESTRICT` means the parent row cannot go while a child still refers to it. If we genuinely wanted *C25* gone we would have to deal with the orders first.

## The CASCADE Option

`CASCADE` is the other common choice. It says: if the parent goes, take the children with it.

To see the difference, we build a small pair of tables that mirror the customer and order relationship, except this time the foreign key is declared `ON DELETE CASCADE`.

In [ ]:
create_cascade_demo_query = """
DROP TABLE IF EXISTS order_cascade_demo;
DROP TABLE IF EXISTS customer_cascade_demo;
CREATE TABLE customer_cascade_demo (
    Customer_Num VARCHAR(10) PRIMARY KEY,
    Customer_LastName VARCHAR(50)
);
CREATE TABLE order_cascade_demo (
    Order_Num VARCHAR(10) PRIMARY KEY,
    Customer_Num VARCHAR(10),
    Order_Carrier VARCHAR(50),
    FOREIGN KEY (Customer_Num) REFERENCES customer_cascade_demo (Customer_Num)
        ON UPDATE CASCADE
        ON DELETE CASCADE
);
"""
for statement in create_cascade_demo_query.strip().split(";"):
    if statement.strip():
        cursor.execute(statement)

cursor.execute("INSERT INTO customer_cascade_demo VALUES (%s, %s);", ("C25", "Felton"))
cursor.execute("INSERT INTO order_cascade_demo VALUES (%s, %s, %s);", ("A1", "C25", "UPS Ground"))
cursor.execute("INSERT INTO order_cascade_demo VALUES (%s, %s, %s);", ("A2", "C25", "DHL Ground"))
conn.commit()

run("orders before deleting the customer", "SELECT * FROM order_cascade_demo;")

Now the same delete that `RESTRICT` refused a moment ago. This time it succeeds, and watch what happens to the two orders we never mentioned.

In [ ]:
cursor.execute("DELETE FROM customer_cascade_demo WHERE Customer_Num = %s;", ("C25",))
conn.commit()
print(f"customers deleted: {cursor.rowcount}")

run("orders after deleting the customer", "SELECT * FROM order_cascade_demo;")

Both orders are gone, and we never wrote a `DELETE` against *order_cascade_demo* at all. The database removed them because the foreign key told it to.

That is the whole point of the option. `RESTRICT` protects you from losing data you did not mean to lose. `CASCADE` saves you from having to clean up by hand, and will quietly delete a great deal if you point it at the wrong row. Neither is the right answer in general, which is why you have to pick one when you design the table.

For reference, you can read back the choice that was recorded for any foreign key.

In [ ]:
rules_query = """
SELECT TABLE_NAME, CONSTRAINT_NAME, UPDATE_RULE, DELETE_RULE
FROM information_schema.REFERENTIAL_CONSTRAINTS
WHERE CONSTRAINT_SCHEMA = %s
ORDER BY TABLE_NAME;
"""
run("the ON UPDATE / ON DELETE rule on every foreign key", rules_query, (DB_NAME,))

## Cleaning Up

The demo tables have done their job, so we drop them. Dropping *customer_cascade_demo* has to wait until the table that references it is gone, for the same reason the earlier delete was refused.

The real tables are untouched. The only statement we ever aimed at them was the one that failed.

In [ ]:
for table in ["order_cascade_demo", "customer_cascade_demo", "customer_demo", "vendor_demo"]:
    cursor.execute(f"DROP TABLE IF EXISTS {table};")
conn.commit()

cursor.execute("SELECT COUNT(*) FROM customer;")
print(f"customer still holds {cursor.fetchone()[0]} rows, exactly as we found it.")

### Closing Up

And we close *cursor* and then *conn*, same as every other tutorial.

In [ ]:
cursor.close()
conn.close()